In [ ]:
# read and plot the data after application exits
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib widget

# Load CSV (skip first 2 header rows, use row 2 as column names)
df = pd.read_csv("data.csv", skiprows=2)

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Position trace (trajectory)
axes[0, 0].plot(df["mouseX"], df["mouseY"], "b.-", alpha=0.6, linewidth=0.5)
axes[0, 0].set_xlabel("X Position (px)")
axes[0, 0].set_ylabel("Y Position (px)")
axes[0, 0].set_title("Cursor Trajectory")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axis("equal")

# 2. Pressure over time
df["time_elapsed"] = (
    df["event_timestamp"] - df["event_timestamp"].iloc[0]
) / 1000  # seconds
axes[0, 1].plot(df["time_elapsed"], df["pressure"], "k-", linewidth=1)
# plot the df[pressure_low] and pressure_high as lines on the pressure plot
axes[0, 1].plot(df["time_elapsed"], df["pressure_low"], color="g", linestyle="--", label="Pressure Band Low")
axes[0, 1].plot(df["time_elapsed"], df["pressure_high"], color="r", linestyle="--", label="Pressure Band High")

# plot the pressure in green if it's in between pressure_low and pressure_high, otherwise in red
for i in range(len(df)):
    if df["pressure_low"].iloc[i] <= df["pressure"].iloc[i] <= df["pressure_high"].iloc[i]:
        axes[0, 1].plot(df["time_elapsed"].iloc[i], df["pressure"].iloc[i], "go", alpha=0.7)
    else:
        axes[0, 1].plot(df["time_elapsed"].iloc[i], df["pressure"].iloc[i], "ro", alpha=0.7)

axes[0, 1].set_xlabel("Time (s)")
axes[0, 1].set_ylabel("Pressure")
axes[0, 1].set_title("Tablet Pressure Over Time")
axes[0, 1].grid(True, alpha=0.3)

# 3. x and y positions over time
axes[1, 0].plot(df["time_elapsed"], df["mouseX"], ".-", label="X Position", alpha=0.7)
axes[1, 0].plot(df["time_elapsed"], df["mouseY"], ".-", label="Y Position", alpha=0.7)
axes[1, 0].set_xlabel("Time (s)")
axes[1, 0].set_ylabel("Position (px)")
axes[1, 0].set_title("Cursor Position Over Time")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Tilt angles over time
axes[1, 1].plot(df["time_elapsed"], df["tiltX"], ".-", label="Tilt X", alpha=0.7)
axes[1, 1].plot(df["time_elapsed"], df["tiltY"], ".-", label="Tilt Y", alpha=0.7)
axes[1, 1].set_xlabel("Time (s)")
axes[1, 1].set_ylabel("Tilt Angle (degrees)")
axes[1, 1].set_title("Stylus Tilt Over Time")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# create a histogram of the latencies between event_timestamp and call_time and a boxplot
# of the latencies in a figure with two subplots that are on top of each other and share
# the x axis, with the boxplot on the top ann no box around the histogram
df["latency_ms"] = (
    df["unix_timestamp"]
    - df["event_timestamp"]
    - np.min(df["unix_timestamp"] - df["event_timestamp"])
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
# Boxplot
ax1.boxplot(df["latency_ms"], vert=False)
ax1.set_title("Latency Boxplot")
ax1.set_yticks([])  # Hide y-axis ticks
# show the mean and median on the boxplot
mean_latency = np.mean(df["latency_ms"])
median_latency = np.median(df["latency_ms"])
ax1.axvline(mean_latency, color="r", label=f"Mean: {mean_latency:.2f} ms")
ax1.axvline(median_latency, color="g", label=f"Median: {median_latency:.2f} ms")
ax1.legend()


# Histogram
ax2.hist(df["latency_ms"], bins=50, color="skyblue", edgecolor="black")
ax2.set_title("Latency Histogram")
ax2.set_xlabel("Latency (ms)")
ax2.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# compute velocity for x y positions using event_timestamp and call_time with gradient
df["v_x_event"] = np.gradient(df["mouseX"]) / np.gradient(
    df["event_timestamp"] / 1000
)  # px/s
df["v_y_event"] = np.gradient(df["mouseY"]) / np.gradient(
    df["event_timestamp"] / 1000
)  # px/s
df["v_event"] = np.sqrt(df["v_x_event"] ** 2 + df["v_y_event"] ** 2)

df["v_x_call"] = np.gradient(df["mouseX"]) / np.gradient(
    df["unix_timestamp"] / 1000
)  # px/s
df["v_y_call"] = np.gradient(df["mouseY"]) / np.gradient(
    df["unix_timestamp"] / 1000
)  # px/s
df["v_call"] = np.sqrt(df["v_x_call"] ** 2 + df["v_y_call"] ** 2)

# comput acceleration for both velocities
df["a_event"] = np.gradient(df["v_event"]) / np.gradient(
    df["event_timestamp"] / 1000
)  # px/s^2
df["a_call"] = np.gradient(df["v_call"]) / np.gradient(
    df["unix_timestamp"] / 1000
)  # px/s^2

# plot the two velocities on the same graph and the two accelerations below them
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
# Velocities
ax1.plot(
    df["time_elapsed"], df["v_event"], label="Velocity (event_timestamp)", alpha=0.7
)
ax1.plot(df["time_elapsed"], df["v_call"], label="Velocity (unix_timestamp)", alpha=0.7)
ax1.set_title("Cursor Velocity Over Time")
ax1.set_ylabel("Velocity (px/s)")
ax1.legend()
ax1.grid(True, alpha=0.3)
# Accelerations
ax2.plot(
    df["time_elapsed"], df["a_event"], label="Acceleration (event_timestamp)", alpha=0.7
)
ax2.plot(
    df["time_elapsed"], df["a_call"], label="Acceleration (unix_timestamp)", alpha=0.7
)
ax2.set_title("Cursor Acceleration Over Time")
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Acceleration (px/s²)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# get the sampling rate based on event_timestamp and call_time
df["dt_event"] = np.gradient(df["event_timestamp"] / 1000)
df["dt_call"] = np.gradient(df["unix_timestamp"] / 1000)
# check for zero dt values and replace them with nan to avoid division by zero
df.loc[df["dt_event"] == 0, "dt_event"] = np.nan
df.loc[df["dt_call"] == 0, "dt_call"] = np.nan
# compute sampling frequency
df["fs_event"] = 1 / df["dt_event"]
df["fs_call"] = 1 / df["dt_call"]

# remove nan values from fs_event and fs_call for boxplot
fs_event_clean = df["fs_event"].dropna()
fs_call_clean = df["fs_call"].dropna()
nb_nan_event = len(df["fs_event"]) - len(fs_event_clean)
nb_nan_call = len(df["fs_call"]) - len(fs_call_clean)

# Calculate statistics
dt_event_clean = df["dt_event"].dropna()
dt_call_clean = df["dt_call"].dropna()

# Calculate recording duration
duration_event = (
    df["event_timestamp"].iloc[-1] - df["event_timestamp"].iloc[0]
) / 1000  # in seconds
duration_call = (
    df["unix_timestamp"].iloc[-1] - df["unix_timestamp"].iloc[0]
) / 1000  # in seconds
n_samples = len(df)
mean_fs_event = n_samples / duration_event
mean_fs_call = n_samples / duration_call

# Create a single figure with 6 subplots (3 rows, 2 columns)
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# Row 1: Delta Time Boxplots
axes[0, 0].boxplot(dt_event_clean, vert=False)
axes[0, 0].axvline(
    dt_event_clean.mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {dt_event_clean.mean():.4f}",
)
axes[0, 0].axvline(
    dt_event_clean.median(),
    color="green",
    linestyle="--",
    linewidth=2,
    label=f"Median: {dt_event_clean.median():.4f}",
)
axes[0, 0].set_title(
    f"Delta Time (gradient of event_timestamp)\n{n_samples} samples in {duration_event:.3f}s, {mean_fs_event:.3f} Hz"
)
axes[0, 0].set_xlabel("Delta Time (s)")
axes[0, 0].legend()

axes[0, 1].boxplot(dt_call_clean, vert=False)
axes[0, 1].axvline(
    dt_call_clean.mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {dt_call_clean.mean():.4f}",
)
axes[0, 1].axvline(
    dt_call_clean.median(),
    color="green",
    linestyle="--",
    linewidth=2,
    label=f"Median: {dt_call_clean.median():.4f}",
)
axes[0, 1].set_title(
    f"Delta Time (gradient of call_time)\n{n_samples} samples in {duration_call:.3f}s, {mean_fs_call:.3f} Hz"
)
axes[0, 1].set_xlabel("Delta Time (s)")
axes[0, 1].legend()

# Row 2: Sampling Rate Histograms
axes[1, 0].hist(df["fs_event"], bins=50, color="lightgreen", edgecolor="black")
axes[1, 0].set_title(
    f"Sampling Rate (gradient of event_timestamp)\nNaNs removed: {nb_nan_event}"
)
axes[1, 0].set_xlabel("Sampling Rate (Hz)")
axes[1, 0].set_ylabel("Frequency")

axes[1, 1].hist(df["fs_call"], bins=50, color="lightcoral", edgecolor="black")
axes[1, 1].set_title(
    f"Sampling Rate (gradient of call_time)\nNaNs removed: {nb_nan_call}"
)
axes[1, 1].set_xlabel("Sampling Rate (Hz)")
axes[1, 1].set_ylabel("Frequency")

# Row 3: Sampling Rate Boxplots
axes[2, 0].boxplot(fs_event_clean, vert=False)
axes[2, 0].axvline(
    fs_event_clean.mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {fs_event_clean.mean():.1f} Hz",
)
axes[2, 0].axvline(
    fs_event_clean.median(),
    color="green",
    linestyle="--",
    linewidth=2,
    label=f"Median: {fs_event_clean.median():.1f} Hz",
)
axes[2, 0].set_xlabel("Sampling Rate (Hz)")
axes[2, 0].legend()

axes[2, 1].boxplot(fs_call_clean, vert=False)
axes[2, 1].axvline(
    fs_call_clean.mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {fs_call_clean.mean():.1f} Hz",
)
axes[2, 1].axvline(
    fs_call_clean.median(),
    color="green",
    linestyle="--",
    linewidth=2,
    label=f"Median: {fs_call_clean.median():.1f} Hz",
)
axes[2, 1].set_xlabel("Sampling Rate (Hz)")
axes[2, 1].legend()

# add a suptitle to the figure
if df["pressure"].max() > 0:
    fig.suptitle(
        f"Detailed Analysis of {n_samples} samples in {duration_event:.3f}s: {mean_fs_event:.6f} Hz, {1000/mean_fs_event:.3f} ms\n Tablet Input Detected\n",
        fontsize=16,
    )
else:
    fig.suptitle(
        f"Detailed Analysis of {n_samples} samples in {duration_event:.3f}s: {mean_fs_event:.6f} Hz, {1000/mean_fs_event:.3f} ms\n Mouse Input Detected\n",
        fontsize=16,
    )

plt.tight_layout()
plt.show()